# 🎓 AWS Learning Assistant - PHASE 1 成果分析

**実装日**: 2026年2月28日  
**ステータス**: ✅ PHASE 1 完了  
**テストケース**: 東大2025年 数学第1問（不定積分）

このノートブックでは、PHASE 1 で実装した教材生成機能の成果を可視化・分析します。

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from datetime import datetime
from pathlib import Path

# 日本語フォント設定
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ ライブラリのインポート完了")

##  📂 PHASE 1 生成結果の読み込み

In [ ]:
# JSON ファイルパス
json_file = Path('test_results/learning_material_demo.json')

# JSON データを読み込み
with open(json_file, 'r', encoding='utf-8') as f:
    material_data = json.load(f)

print(f"✅ JSON ファイルを読み込みました: {json_file}")
print(f"\n📋 教材ID: {material_data['materialId']}")
print(f"📅 作成日時: {material_data['createdAt']}")
print(f"🎓 難易度: {material_data['difficulty_level']}")
print(f"🏛️  大学: {material_data['examMetadata']['university']} {material_data['examMetadata']['year']}年")

## 📊 成果メトリクスの分析

In [ ]:
# メトリクスを計算
metrics = {
    '項目': [
        'アウトラインステップ数',
        '重要概念数',
        'クイズ問題数',
        'よくある間違い数',
        '参考問題数',
        '解答信頼度',
        '難易度レベル',
        'AWS Personalization スコア'
    ],
    '値': [
        len(material_data['outline']),
        len(material_data['keyConcepts']),
        len(material_data['quizQuestions']),
        len(material_data['commonMistakes']),
        len(material_data['referenceProblems']),
        f"{material_data['solutionConfidence']:.0%}",
        material_data['difficulty_level'],
        f"{material_data['awsPersonalizationScore']:.2f}"
    ]
}

metrics_df = pd.DataFrame(metrics)
print("\n📊 PHASE 1 生成メトリクス")
print("="*60)
print(metrics_df.to_string(index=False))
print("="*60)

### 🎯 問題と生成解答

In [ ]:
print("\n📝 問題")
print("-" * 60)
print(material_data['problemText'])
print("\n✅ 解答")
print("-" * 60)
print(f"最終答: {material_data['solutionFinal']}")
print(f"LaTeX: {material_data['solutionLatex']}")
print(f"信頼度: {material_data['solutionConfidence']:.0%}")
print(f"OCR提供者: {material_data['ocrProvider']}")

print("\n📋 解答ステップ数:", len(material_data['solutionSteps']))
for i, step in enumerate(material_data['solutionSteps'], 1):
    print(f"  {i}. {step[:80]}...")

## 📈 ビジュアライゼーション

In [ ]:
### グラフ1: 生成教材の構成成分

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左図: 円グラフ
components = {
    'Outline Steps': len(material_data['outline']),
    'Key Concepts': len(material_data['keyConcepts']),
    'Quiz Questions': len(material_data['quizQuestions']),
    'Common Mistakes': len(material_data['commonMistakes']),
    'Reference Problems': len(material_data['referenceProblems'])
}

colors = ['#667eea', '#764ba2', '#f093fb', '#4facfe', '#00f2fe']
ax1.pie(components.values(), labels=components.keys(), autopct='%1.0f%%', colors=colors, startangle=90)
ax1.set_title('Generated Material Components', fontsize=12, fontweight='bold')

# 右図: 棒グラフ
component_names = list(components.keys())
component_values = list(components.values())
bars = ax2.bar(component_names, component_values, color=colors)
ax2.set_ylabel('Count', fontweight='bold')
ax2.set_title('Component Quantities', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# 棒の上に数字を表示
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}', ha='center', va='bottom', fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("✅ グラフ1 完成")

In [ ]:
### グラフ2: 信頼度と品質スコア

fig, ax = plt.subplots(figsize=(10, 6))

metrics_names = ['Solution Confidence', 'AWS Personalization Score']
metrics_values = [
    material_data['solutionConfidence'] * 100,
    material_data['awsPersonalizationScore'] * 100
]

colors_bar = ['#4caf50', '#667eea']
bars = ax.barh(metrics_names, metrics_values, color=colors_bar, height=0.4)

# パーセンテージ表示
for i, (bar, value) in enumerate(zip(bars, metrics_values)):
    ax.text(value + 2, bar.get_y() + bar.get_height()/2,
            f'{value:.0f}%', va='center', fontweight='bold', fontsize=12)

ax.set_xlim(0, 110)
ax.set_xlabel('Score (%)', fontweight='bold')
ax.set_title('Quality Metrics', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ グラフ2 完成")

### 🗂️ 生成教材の詳細内容

In [ ]:
### 📚 アウトライン（学習ステップ）
print("\n📚 学習アウトライン:")
for step in material_data['outline']:
    print(f"  {step['stepNumber']}. {step['brief']}")

### 🔑 重要概念
print("\n\n🔑 重要概念:")
for concept in material_data['keyConcepts']:
    print(f"  • {concept}")

### 🎓 学習目標
print("\n\n🎓 学習目標:")
for i, obj in enumerate(material_data['learningObjectives'], 1):
    print(f"  {i}. {obj}")

### ❌ よくある間違い
print("\n\n❌ よくある間違い:")
for mistake in material_data['commonMistakes']:
    print(f"  • {mistake['mistakeDescription']}")
    print(f"    理由: {mistake['whyWrong']}")
    print(f"    対策: {mistake['correction']}")

## ✅ PHASE 1 完成サマリー

In [ ]:
print("""
✨================================================================================✨
                🎓 AWS Learning Assistant - PHASE 1 完成報告 🎓
================================================================================

🎯 実装内容:
  ✅ LearningMaterial データモデル（6つのサブモデル含む）
  ✅ MaterialGenerator 教材生成エンジン
  ✅ FastAPI Learning Routes（7つのエンドポイント）
  ✅ デモテストスクリプト（実行・成功）
  ✅ 統合テスト実装
  ✅ JSONスキーマ検証
  ✅ HTMLダッシュボード作成
  ✅ Jupyterノートブック分析ツール

📊 生成結果統計:
  • Material ID: {material_data['materialId'][:12]}...
  • 難易度: {material_data['difficulty_level']}
  • アウトラインステップ: {len(material_data['outline'])}
  • 重要概念: {len(material_data['keyConcepts'])}
  • クイズ問題: {len(material_data['quizQuestions'])}
  • よくある間違い: {len(material_data['commonMistakes'])}
  • 参考問題: {len(material_data['referenceProblems'])}
  • 解答信頼度: {material_data['solutionConfidence']:.0%}
  • AWS Personalization スコア: {material_data['awsPersonalizationScore']:.2f}

🚀 PHASE 2 ロードマップ:
  🔜 Bedrock 統合 - Claude 3.5 Sonnet で解説拡張
  🔜 Polly 統合 - Neural Voice で音声化
  🔜 Personalize 統合 - ML推薦エンジン
  🔜 React UI 拡張 - フロントエンド統合

📁 成果ファイル:
  • docs/AWS_LEARNING_ASSISTANT_PHASE1_COMPLETION.md (実装ドキュメント)
  • docs/PHASE1_DASHBOARD.html (ビジュアルダッシュボード)
  • services/api/PHASE1_Analysis.ipynb (このノートブック)
  • services/api/test_results/learning_material_demo.json (生成JSON)

✨================================================================================✨
""".format(material_data=material_data))